# Chapter 04: The Architecture — From Qubits to Systems

## Introduction

Welcome to **Section 1.3: General Lecture on Quantum Technology**.

We have a qubit.
We have a gate.
Now we need a **Computer**.

A pile of transistors is not a laptop.
A pile of qubits is not a Quantum Computer.
We need an architecture.
We need a way to organize them.
We need a way to wire them up.
We need a way to control them all at once.

**The Structure of this Notebook:**
1.  **The Von Neumann Bottleneck:** Why classical architecture fails.
2.  **The Quantum Stack:** From Python to Pulse.
3.  **Connectivity:** Who can talk to whom?
4.  **The Wiring Nightmare:** 5000 cables in a fridge.
5.  **Control Electronics:** FPGAs and Cryo-CMOS.
6.  **QRAM:** The memory problem.
7.  **Distributed Quantum Computing:** The Quantum Internet.
8.  **Simulation:** Testing connectivity graphs.
9.  **Surface Code Architecture:** Logical Qubits.
10. **Modular Architecture:** The only way to scale.
11. **The Control Plane:** FPGA logic.
12. **Quantum Error Correction (Stabilizers):** The math of protection.
13. **The Roadmaps:** IBM, Google, PsiQuantum.
14. **The Economics:** Why is this expensive?
15. **The Energy Crisis:** Power Consumption.
16. **Transpilation Deep Dive:** Mapping circuits.
17. **Python Exercise:** Simulating Surface Code.

Let us begin.

---

## Part 1: The Von Neumann Bottleneck

**Classical Architecture**
In your laptop, the CPU and RAM are separate.
To add two numbers, the CPU fetches them from RAM.
It adds them.
It writes the result back to RAM.
The bus between them is the bottleneck.

**Quantum Architecture (QPU)**
In a QPU, the Processor IS the Memory.
The state is stored in the qubits.
The operations are performed ON the qubits.
There is no "Fetch".
There is no "Store".
There is only "Evolve".

**The Hybrid Model**
A quantum computer is not a standalone device.
It is a **Co-Processor**.
Like a GPU.
The CPU runs the main program (Python).
When it hits a hard math problem, it sends a job to the QPU.
The QPU runs the circuit 1000 times.
It returns the counts: `{'00': 500, '11': 500}`.
The CPU continues.

---

## Part 2: The Quantum Stack

**Level 1: Application (Python)**
User writes: `q = QuantumRegister(2); qc.h(q[0]); qc.cx(q[0], q[1])`.
Frameworks: Qiskit, Cirq, Pennylane.

**Level 2: Intermediate Representation (QASM)**
The code is compiled to OpenQASM.
This is the assembly language.
`h q[0]; cx q[0], q[1];`.

**Level 3: Transpilation (Logical to Physical)**
The compiler maps the logical qubits to physical qubits.
It adds SWAP gates if they are not connected.
It optimizes the circuit (cancels adjacent gates).

**Level 4: Pulse Schedule (OpenPulse)**
The gates are converted to microwave pulses.
Defined by Frequency, Amplitude, Phase, Duration.

**Level 5: Hardware (AWG)**
The Arbitrary Waveform Generator (AWG) plays the pulses.
The signals go down the fridge.
They hit the qubit.

---

## Part 3: Connectivity (Topology)

**The Problem**
In a classical CPU, any transistor can talk to any other (via the bus).
In a QPU, interactions are local.
Qubit 1 can only talk to Qubit 2 if there is a physical wire between them.

**Topology 1: Linear Chain**
Common in Trapped Ions.
0 - 1 - 2 - 3 - 4.
To entangle 0 and 4, you need to SWAP 0 all the way down.
Cost: $2N$ gates.

**Topology 2: Square Grid (Google Sycamore)**
Each qubit has 4 neighbors.
Good for Surface Code error correction.
Bad for Crosstalk (too many connections).

**Topology 3: Heavy Hex (IBM Eagle/Osprey)**
Each qubit has only 2 or 3 neighbors.
It looks like a honeycomb.
Reduces crosstalk significantly.
But requires more SWAP gates to move data.

**Topology 4: All-to-All (IonQ)**
Every qubit is connected to every other qubit.
Achieved by moving the ions physically.
Hardware efficient, but slow.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# PART 3 CODE: VISUALIZING TOPOLOGIES

def draw_topology(graph, title):
    plt.figure(figsize=(6, 6))
    pos = nx.spring_layout(graph, seed=42)
    nx.draw(graph, pos, with_labels=True, node_color='lightblue', 
            node_size=800, font_weight='bold', edge_color='gray')
    plt.title(title)
    plt.show()

# 1. Linear Chain
G_linear = nx.path_graph(10)
# draw_topology(G_linear, "Linear Chain (Trapped Ion)")

# 2. Grid (Sycamore)
G_grid = nx.grid_2d_graph(4, 4)
# draw_topology(G_grid, "Square Grid (Google Sycamore)")

# 3. Heavy Hex (Approximation)
G_hex = nx.hexagonal_lattice_graph(2, 2)
draw_topology(G_hex, "Hexagonal Lattice (IBM Concept)")


---

## Part 4: The Wiring Nightmare

**The Scale Problem**
IBM Condor has 1121 qubits.
If each needs 2 lines (Input/Output), that is 2242 cables.
These are coaxial cables.
They are thick.
They conduct heat.
The fridge has a cooling power of 10 micro-Watts at 15mK.
If you add too many cables, the heat load leaks in.
The temperature rises.
The qubits die.

**Multiplexing**
Solution 1: Frequency Multiplexing.
Send 10 signals down 1 cable at different frequencies.
Like radio stations.
Used for Readout.
Hard for Control (intermodulation distortion).

**Flex Cables**
Solution 2: Replace rigid coax with flexible ribbon cables.
Like the ones in your laptop.
Made of Kapton and Superconductors.
Google uses this.

---

## Part 5: Control Electronics (Checkmate)

**Room Temperature Control**
Typically, the FPGA logic is at 300K (Room Temp).
The signals travel down 2 meters of cable to 15mK.
The latency is ~200 nanoseconds round trip.
This is too slow for fast feedback (Error Correction).

**Cryo-CMOS (Horse Ridge)**
Intel built a control chip that lives inside the fridge at 4 Kelvin.
It generates the pulses locally.
This reduces the latency to < 10ns.
It also reduces the cable count.
You just need 1 fiber to send instructions to the Cryo-Chip.
The Cryo-Chip handles the 1000 qubits.
This is the future.
But designing CMOS that works at 4K is hard (transistor thresholds change).

---

## Part 6: QRAM (The Memory Problem)

**Why is Machine Learning hard?**
In ML, you load a dataset (Images, Text).
You process it.
In Quantum ML, loading the data is $O(N)$.
Processing is $O(\sqrt{N})$.
The loading kills the speedup.
We need a QRAM (Quantum RAM).
A device where we can query an address $|i\rangle$ and get the data $|d_i\rangle$ in superposition.

**The Bucket Brigade Architecture**
Proposed by Seth Lloyd.
A binary tree of switches.
To acccess memory address 101, you route the photon Left-Right-Left.
It requires $O(N)$ active switches.
It has high error rates.
Nobody has built a large QRAM yet.
This is the biggest bottleneck for "Big Data" quantum applications.

---

## Part 7: Distributed Quantum Computing

**The Chandelier Limit**
A dilution fridge can only be so big.
Maybe we can fit 10,000 qubits.
We cannot fit 1,000,000.
So we need multiple fridges.

**Optical Interconnects**
How do you connect two fridges?
Microwaves cannot leave the fridge (thermal noise destroys them).
We must convert Microwaves to Optics (Light).
We need a Transducer.
A device that converts a 5 GHz electrical photon to a 193 THz optical photon.
This is extremely hard (efficiency is currently < 1%).
But if we solve it, we can build the **Quantum Internet**.

---

## Part 8: Simulation: SWAP Overhead

Let's write a script to check how many SWAPs we need for a CNOT.
We will use NetworkX to find the shortest path.

In [ ]:
import networkx as nx

def calculate_swap_cost(graph, qubit_1, qubit_2):
    if qubit_1 == qubit_2:
        return 0
    
    # Shortest path
    try:
        path = nx.shortest_path(graph, qubit_1, qubit_2)
    except nx.NetworkXNoPath:
        return 9999 # Disconnected
    
    # Distance
    distance = len(path) - 1
    
    # To interact, we need (Distance - 1) SWAPs to bring them next to each other.
    # Each SWAP is typically 3 CNOTs.
    return (distance - 1) * 3

# Define Architectures
linear = nx.path_graph(20)
grid = nx.grid_2d_graph(4, 5) # 20 qubits

# Test Case: Connect first and last
cost_linear = calculate_swap_cost(linear, 0, 19)
cost_grid = calculate_swap_cost(grid, (0,0), (3,4))

print(f"Cost to connect Q0 to Q19 in Linear Chain: {cost_linear} CNOTs")
print(f"Cost to connect Q0 to Q19 in Grid: {cost_grid} CNOTs")
print("Conclusion: Grids are much better than Chains for long-range connectivity.")

---

## Part 9: Surface Code Architecture

**The Concept**
We don't trust physical qubits.
We group them into a tile.
Example: 9 physical qubits = 1 logical qubit.
Data Qubits store the information.
Ancilla Qubits check for errors (Parity checks).
This creates a checkerboard pattern.

**The Threshold**
If physical error < 1%, the logical error goes down as we add more qubits.
If physical error > 1%, the logical error goes UP.
We are currently right at the threshold (Google 2023 Result).

**The Cost**
To break RSA-2048, we need 4000 logical qubits.
With Surface Code, each logical qubit needs 1000 physical qubits.
Total needed: 4,000,000 physical qubits.
Current status: 1,000 qubits.
We have a long way to go.

---

## Part 10: Modular Architecture (Lizard Head)

**The Vision**
We will not build one giant chip.
We will build Modules.
Each module has 1000 qubits and its own control electronics.
We connect them like LEGO bricks.
Short range connections: Capacitive bridges.
Long range connections: Optical fibers.

**IBM Kookaburra**
This is IBM's plan for 2026.
Multichip modules.
Couplers that connect chips together seamlessly.
This allows them to scale to 10k, 100k qubits.

---

## Part 11: The Control Plane

**Classical Logic**
We need fast classical logic.
If we detect an error (Measure Ancilla = 1), we must correct it.
This requires an IF-THEN statement.
IF error, apply X gate.
This must happen within nanoseconds.

**FPGA (Field Programmable Gate Array)**
This is why we use FPGAs, not CPUs.
FPGAs are hardware-programmable.
They can make decisions in 10-20ns.
A CPU would take microseconds (too slow).
The Control Plane is the brain of the quantum computer.
The qubits are just the muscle.

---

## Part 12: Quantum Error Correction (The Stabilizers)

**The Stabilizer Formalism**
We define a code by the operators that "stabilize" the valid states.
Example: 2-qubit repetition.
$$ S = Z_1 Z_2 $$
If the state is $|00\rangle$, then $Z_1 Z_2 |00\rangle = (+1)(+1)|00\rangle = |00\rangle$.
If the state is $|11\rangle$, then $Z_1 Z_2 |11\rangle = (-1)(-1)|11\rangle = |11\rangle$.
So $|00\rangle$ and $|11\rangle$ are stable (+1 eigenvalue).
If an error happens ($X_1$), state becomes $|10\rangle$.
$Z_1 Z_2 |10\rangle = (-1)(+1)|10\rangle = -|10\rangle$.
The measurement gives -1.
This is the **Syndrome**.
We know something broke.

**X-Stabilizers and Z-Stabilizers**
Z-stabilizers detect Bit-flip errors.
X-stabilizers detect Phase-flip errors.
For the Surface Code, we tile the plane with X and Z checks.
It works like a Minesweeper game.
We sweep the board for mines (errors) and flag them.

---

## Part 13: Industry Roadmaps

**IBM (The Superconducting Path)**
-   2023: Osprey (433 qubits).
-   2024: Condor (1121 qubits).
-   2025: Flamingo (1386 qubits, Modular).
-   2026: Kookaburra (4158 qubits, Multichip).
-   Goal: 100,000 qubit system by 2033 (Centennial).

**Google (The Surface Code Path)**
-   Focus on QUALITY, not quantity.
-   Milestone 1: Quantum Supremacy (Achieved 2019).
-   Milestone 2: Logical Qubit > Physical Qubit (Achieved 2023).
-   Milestone 3: Long-lived Logical Qubit.
-   Goal: A 1,000,000 physical qubit machine.

**PsiQuantum (The Photonic Path)**
-   Building a factory.
-   They skip the "NISQ" era.
-   Aiming straight for a Utility-Scale Fault-Tolerant machine.
-   Promised 1,000,000 qubits by mid-decade (aggressive).

---

## Part 14: The Economics of Quantum Computing

**The Cost of Entry**
A single dilution refrigerator costs $500,000.
The control electronics cost another $500,000.
The fabrication facility (Fab) costs $100 Million+.
This is why there are no "Garage Quantum Startups" building hardware.
It is a game for governments and tech giants.

**The Cloud Model**
Because the hardware is so expensive, nobody will own a QC.
We will access it via the Cloud (IBM Quantum Experience, AWS Braket).
You pay per second of runtime.
Currently, it is free for research, expensive for enterprise.

**The Market Size**
BCG predicts a $850 Billion market by 2040.
This value comes not from selling hardware, but from the **Value Created**.
-   New Drugs.
-   Better Batteries.
-   Optimized Logistics.

---

## Part 15: The Energy Crisis

**Is Quantum Green?**
A Supercomputer (Frontier) burns 20 MegaWatts.
A Quantum Computer burns 10 KiloWatts (mostly for the compressor).
So Quantum is much greener per calculation.

**The Cooling Penalty**
However, removing 1 Watt of heat at 15mK requires 10,000 Watts of wall power.
The thermodynamic efficiency is terrible.
As we scale to 1,000,000 qubits, the cooling power required might become unsustainable.
This is why Photonic QC (which works at room temp, mostly) is attractive.

---

## Part 16: Transpilation Deep Dive

**The Compiler Pipeline**
When you hit "Run", Qiskit does this:
1.  **Unrolling:** Break custom gates into basis gates (U1, U2, U3, CX).
2.  **Layout:** Choose WHICH physical qubits to use (map q0 -> Q5, q1 -> Q12).
    -   It picks the best qubits (lowest error rates).
3.  **Routing:** Insert SWAP gates to satisfy connectivity.
    -   This is an NP-Hard problem.
    -   We use heuristic algorithms (Stochastic Swap).
4.  **Optimization:** Merge adjacent rotations.
    -   $R_z(\theta) R_z(\phi) = R_z(\theta + \phi)$.
    -   Cancel $CX - CX = Identity$.

**Sabre Swap**
The state-of-the-art routing algorithm.
It looks ahead at future gates to minimize SWAPs.
Standard in Qiskit.

---

## Part 17: Python Exercise: Simulating Surface Code (Simple)

Let's simulate a simple 3-qubit repetition code.
Bit Flip Code.

In [ ]:
import random

def bit_flip_simulation(shots=1000, error_rate=0.1):
    success = 0
    
    for _ in range(shots):
        # 1. Encode logical 0 -> |000>
        state = [0, 0, 0]
        
        # 2. Add Noise
        for i in range(3):
            if random.random() < error_rate:
                state[i] = 1 - state[i] # Flip it
                
        # 3. Measure Stabilizers (Syndrome Measurement)
        # Z1Z2 and Z2Z3
        # Parity check: (s1 + s2) % 2
        s1 = (state[0] + state[1]) % 2
        s2 = (state[1] + state[2]) % 2
        
        # 4. Correction Logic
        if s1 == 1 and s2 == 0:
            state[0] = 1 - state[0] # Flip Q0 back
        elif s1 == 1 and s2 == 1:
            state[1] = 1 - state[1] # Flip Q1 back
        elif s1 == 0 and s2 == 1:
            state[2] = 1 - state[2] # Flip Q2 back
            
        # 5. Check if logical state is preserved
        # If state is [0,0,0], success.
        # Logic handles single bit flips. Fails on 2-bit flips.
        if state == [0, 0, 0]:
            success += 1
            
    return success / shots

print("Comparing Physical vs Logical Error Rate:")
p_physical = 0.1
logical_fidelity = bit_flip_simulation(shots=10000, error_rate=p_physical)
print(f"Physical Error Rate: {p_physical*100}%")
print(f"Logical Error Rate: {(1-logical_fidelity)*100:.2f}%")
print("It is better! (Because probability of 2 errors is p^2 = 0.01)")

---

## Conclusion

Architecture is where Physics meets Engineering.
Physics says: "Here is a qubit."
Engineering says: "How do I fit 10,000 cables in a 10cm tube?"
The problems we face today are not just Quantum problems.
They are Heat problems.
They are Latency problems.
They are Integration problems.

In the next notebook, we will finally use the machine.
We will look at the question that started it all.
**Quantum Supremacy**.
Can this machine actually beat a Supercomputer?

**Navigation:** [Next → Chapter 05: Quantum Supremacy](05_quantum_supremacy.ipynb)

---

## Appendix: Glossary

-   **Ancilla Qubit:** A helper qubit used to detect errors without destroying data.
-   **Cryo-CMOS:** Classical silicon chips designed to operate at 4 Kelvin.
-   **QRAM:** Quantum Random Access Memory. A hypothetical device to load data in superposition.
-   **Stabilizer Code:** A type of error correction code based on measuring parity checks.
-   **Surface Code:** The leading error correction architecture, using a 2D grid of nearest-neighbor interactions.
-   **Transpiler:** A compiler that converts logical gates to physical pulses, handling connectivity constraints.